# Destiny Explained — Kokoro af_heart, 1×

Épisode 002 : 60 secondes à la Tour. Aucun jeton API requis. Session Python 3.12 / CPU. Les téléchargements du modèle et le rendu peuvent prendre plusieurs minutes. Enregistrer les fichiers avant la fin de session Colab.


In [ ]:
import os, subprocess, sys
from pathlib import Path
assert sys.version_info[:2] == (3, 12), "Utiliser une session Colab Python 3.12"
REF = "feat/kokoro-narration"
PROJECT = Path('/content/Destiny-Explained-Studio')
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch',REF,'--single-branch',
                    'https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(PROJECT)],check=True)
subprocess.run(['git','-C',str(PROJECT),'rev-parse','HEAD'],check=True)


## Installer Node, Python et les bibliothèques Linux
Ces commandes s’exécutent sur Colab, pas sur le téléphone.


In [ ]:
%%bash
set -euo pipefail
cd /content
NODE_VERSION=22.23.4
NODE_ARCHIVE="node-v${NODE_VERSION}-linux-x64.tar.xz"
curl -fsSLO "https://nodejs.org/dist/v${NODE_VERSION}/${NODE_ARCHIVE}"
curl -fsSLo node-SHASUMS256.txt "https://nodejs.org/dist/v${NODE_VERSION}/SHASUMS256.txt"
awk -v f="$NODE_ARCHIVE" '$2 == f' node-SHASUMS256.txt | sha256sum --check -
tar -xJf "$NODE_ARCHIVE"
apt-get update -qq
AUDIO_PACKAGE=libasound2
if apt-cache show libasound2t64 >/dev/null 2>&1; then AUDIO_PACKAGE=libasound2t64; fi
apt-get install -y ffmpeg python3-venv libnss3 libatk-bridge2.0-0 libx11-xcb1 libxcomposite1 libxdamage1 libxrandr2 libgbm1 "$AUDIO_PACKAGE" fonts-liberation
python3 -m venv /content/destiny-tts
/content/destiny-tts/bin/pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cpu
/content/destiny-tts/bin/pip install -r /content/Destiny-Explained-Studio/remotion/tts/requirements-kokoro.txt


## Générer les voix et leurs horodatages
Le rapport affiche les durées mesurées. Une scène trop longue provoque une erreur, sans accélération ou coupure de la voix.


In [ ]:
%%bash
set -euo pipefail
export PATH="/content/destiny-tts/bin:/content/node-v22.23.4-linux-x64/bin:$PATH"
export HF_HUB_DISABLE_XET=1
cd /content/Destiny-Explained-Studio/remotion
npm ci
npm run typecheck
npm run narrate:episode


In [ ]:
import json
from IPython.display import Audio, display
ROOT = PROJECT / 'remotion'
report = json.loads((ROOT / 'out/narration-report.json').read_text())
print('Voix:', report['voice'], 'Vitesse:', report['speed'])
print('Audio synthétisé:', report['speechSeconds'], 's ; vidéo:', report['timelineSeconds'], 's')
for scene in report['scenes']:
    print(scene['id'], scene['audioSeconds'], '/', scene['budgetSeconds'], 's')
display(Audio(filename=str(ROOT / 'public' / report['scenes'][0]['source'])))


## Rendre le MP4 complet avec audio
Le rendu utilise le processeur. Laisser la session active. Aucun workflow GitHub Actions n’est appelé.


In [ ]:
%%bash
set -euo pipefail
export PATH="/content/destiny-tts/bin:/content/node-v22.23.4-linux-x64/bin:$PATH"
cd /content/Destiny-Explained-Studio/remotion
npm run browser:ensure
npm run render:episode:voiced
npm run verify:narration
ffprobe -v error -show_entries stream=codec_name,width,height,duration -of json out/episode-002-traveler.mp4


## Télécharger sur Android
Exécuter les téléchargements séparément si Chrome les bloque.


In [ ]:
from google.colab import files
files.download(str(ROOT / 'out/episode-002-traveler.mp4'))


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
ready_path = ROOT / 'out/traveler_episode_002-ready.json'
ready = json.loads(ready_path.read_text())
assets = (ROOT / 'public' / ready['audio']['voiceover']['source']).parent
archive = ROOT / 'out/traveler-narration.zip'
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    for path in assets.iterdir():
        if path.suffix == '.wav' or path.name.endswith('.words.json') or path.name in ('words.json','manifest.json'):
            z.write(path, path.relative_to(ROOT).as_posix())
    z.write(ready_path, 'out/traveler_episode_002-ready.json')
    z.write(ROOT / 'out/narration-report.json', 'out/narration-report.json')
files.download(str(archive))
